# Stage 4 · Data efficiency: the smallest training subset that keeps the full-data mAP50 (assignment 5.4, with 5.3)

Protocol and expectations were declared before any subset run. This notebook only reads finished runs (their `predictions/{dev,val}.csv.gz`), re-evaluates them with
`auric.evaluation`, recomputes each subset's image list from its config (the selection is deterministic: `auric/data/subsets.py`) and answers four things:

| Question | Where |
|---|---|
| Which subsets exist, and what do they keep (images, instances, class and size composition)? | Stage 2 |
| Does a subset reach **90% of the full-data val mAP50**? (`threshold = 0.9 x mean of the full-data seeds`; guard: dev must pass too) | Stage 3 |
| Is the coverage selection better than a random subset of the same label budget? (paired bootstrap, per class) | Stage 4 |
| What does a doubling of labels buy, overall and per class, and what would +500 labels do (assignment 5.3)? | Stage 5 |

**How to read it.** The val set has 22 images and its mAP50 interval is about +-0.04 on a score of ~0.10, wider than the 10% margin (0.01). A subset "passes" when its point estimate
clears the threshold on val *and* dev; whether the margin is resolved at all is shown by the spread of the full-data seeds and the paired intervals. The smallest subset reported as successful is the smallest one **actually
tested** that passes. No mathematically proven minimum is claimed.

**Integrity.** Selection uses train_core images only. Dev and val never enter a selection; val is reported, dev is the second guard.

In [ ]:
import sys
from pathlib import Path

RUNTIME = "local"     # "local" | "kaggle"
if RUNTIME == "local":
    ROOT = Path("..").resolve()                 # the submission folder (run the notebook from notebooks/)
    SRC_DIR, OUTPUTS = ROOT / "src", ROOT / "results" / "runs"            # OUTPUTS/<arm>/predictions/{dev,val}.csv[.gz]
else:
    SRC_DIR = Path("/kaggle/input/datasets/happyanurag/auric-src-subset")                 # the folder that CONTAINS auric/
    OUTPUTS = Path("/kaggle/input/datasets/happyanurag/auric-outputs/Auric-outputs")      # OUTPUTS/<arm>/predictions/{dev,val}.csv
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
CONFIG_FILE = SRC_DIR / "auric" / "configs" / "SC.yaml"                      # data paths, split, size bins (the arms differ only in `subset`)
ENV_FILE = SRC_DIR / "auric" / "configs" / "env" / f"{RUNTIME}.yaml"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from auric.config import load_config
from auric.data.dataset import index_split, load_annotations, read_class_names
from auric.data.splits import ensure_split
from auric.data.subsets import describe_subset, select_subset
from auric.evaluation import compare, evaluate
from auric.evaluation.efficiency import gain_for_extra_labels, log2_slope, recovery_threshold
from auric.structures import load_detections

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 3)

## Stage 1 · Inputs and data

The arms are listed explicitly. Naming: `SR<budget %>_k<subset seed>` (random), `SC<budget %>` (coverage), `D1_s<training seed>` (full data); budget `12` = 12.5% of the train_core instances.
An arm whose predictions are not in `OUTPUTS` is shown as missing and left out, so the notebook can be run again after each batch of Kaggle runs. `A1_s0` is not a subset arm: it is the other full-data model of the report and appears in the Stage 3 per-class tables for comparison with D1.

In [ ]:
BUDGET = {50: 0.5, 25: 0.25, 12: 0.125}                   # arm-name percentage -> fraction of the train_core instances
RANDOM_ARMS = [(50, 0), (25, 0), (12, 0), (12, 1)]        # (budget %, subset seed); (12, 1) is the optional repeat at the smallest budget
COVERAGE_ARMS = [25, 12, 50]                              # SC50 is optional
ARMS = [("D1_s0", "full", None, 1.0, None), ("D1_s1", "full", None, 1.0, None)]
ARMS += [(f"SR{p}_k{k}", "random", "random", BUDGET[p], k) for p, k in RANDOM_ARMS]
ARMS += [(f"SC{p}", "coverage", "coverage", BUDGET[p], 0) for p in COVERAGE_ARMS]
ARMS = pd.DataFrame(ARMS, columns=["arm", "kind", "method", "budget_frac", "subset_seed"]).set_index("arm")
REFERENCE_ARMS = list(ARMS.index[ARMS.kind == "full"])
CONTEXT = ["A1_s0"]       # full-data model with another recipe (2x up-sampled tiles): shown in the tables next to D1, not part of the subset analysis
MIN_CELL = 20                                             # SC.yaml
BOOTSTRAP, SEED = 1000, 0

In [ ]:
cfg = load_config(CONFIG_FILE, ENV_FILE)
DATA_ROOT, SPLIT_FILE, EC = Path(cfg["paths"]["data_root"]), Path(cfg["paths"]["split_file"]), cfg["evaluation"]
NAMES = read_class_names(DATA_ROOT)
SHORT = [n.replace("Truck", "").replace("w/", "").strip() for n in NAMES]
SIZE_BINS = EC["size_bins"]

RECORDS = {s: index_split(DATA_ROOT, s) for s in ("train", "val")}
ANNS = {s: load_annotations(RECORDS[s], len(NAMES)) for s in RECORDS}
split, _ = ensure_split(SPLIT_FILE, ANNS["train"], NAMES, **cfg["split"])
CORE = split["train_core"]
GTS = {"dev": {i: ANNS["train"][i] for i in split["dev"]}, "val": {r.image_id: ANNS["val"][r.image_id] for r in RECORDS["val"]}}


def predictions_file(arm, part):
    # outputs/<run>/<run>_predictions/ ; the first runs (D1, A1) were stored under the experiment name, without the seed suffix
    for name in (arm, arm[:-3]) if arm.endswith("_s0") else (arm,):
        for p in (OUTPUTS / name / "predictions" / f"{part}.csv", OUTPUTS / name / "predictions" / f"{part}.csv.gz",
                  OUTPUTS / name / f"{name}_predictions" / f"{part}.csv", OUTPUTS / name / f"{name}_predictions" / f"{part}.csv.gz"):
            if p.is_file():
                return p


ARMS["available"] = [all(predictions_file(a, p) for p in ("dev", "val")) for a in ARMS.index]
AVAIL = list(ARMS.index[ARMS.available])
CONTEXT = [a for a in CONTEXT if all(predictions_file(a, p) for p in ("dev", "val"))]
display(ARMS.assign(budget_pct=lambda d: 100 * d.budget_frac).drop(columns="budget_frac"))
print(f"{len(AVAIL)} of {len(ARMS)} arms have predictions; train_core: {len(CORE)} images, "
      f"{sum(len(ANNS['train'][i]) for i in CORE)} instances; dev {len(GTS['dev'])} images, val {len(GTS['val'])} images")

## Stage 2 · What each subset keeps

Selections are recomputed from the arm's config values and are deterministic, so this table is reproducible without the run. Read it for: images per label budget (coverage keeps many more scenes
than a random subset of the same budget), the rare-class instances that survive, and the size mix. A random subset at a small budget can miss a class almost entirely.

In [ ]:
COMPOSITION = []
for arm, a in ARMS.iterrows():
    if a.kind == "full":
        ids = [i for i in CORE if len(ANNS["train"][i])]
        sel = {"ids": ids, "instances": sum(len(ANNS["train"][i]) for i in ids)}
    else:
        sel = select_subset(ANNS["train"], CORE, NAMES, SIZE_BINS, a.method, a.budget_frac, int(a.subset_seed), MIN_CELL)
    d = describe_subset(ANNS["train"], CORE, sel["ids"], NAMES, SIZE_BINS)
    row = {"arm": arm, "kind": a.kind, "budget %": 100 * a.budget_frac, "images": len(sel["ids"]), "instances": sel["instances"],
           "median inst/image": d["images"].loc["subset", "instances / image (median)"], "xs %": d["sizes"]["share % (subset)"].iloc[0]}
    row.update({f"n {s}": int(v) for s, v in zip(SHORT, d["classes"]["instances (subset)"])})
    row.update({f"img {s}": int(v) for s, v in zip(SHORT, d["classes"]["images with class (subset)"])})
    COMPOSITION.append(row)
COMPOSITION = pd.DataFrame(COMPOSITION).set_index("arm")
N_FULL = float(COMPOSITION.loc[REFERENCE_ARMS[0], "instances"])
display(COMPOSITION[COMPOSITION.kind != "full"].round(1))

## Stage 3 · Does a subset recover 90% of the full-data mAP50?

`threshold = 0.9 x mean(val mAP50 of the full-data seeds)`; the same on dev. The spread of the full-data seeds (min / max) is the seed noise against which the 10% margin is read: if it is as wide as the margin,
"passes" cannot be told from "equal by chance".

In [ ]:
REPORT, DETS = {}, {}
for arm in AVAIL + CONTEXT:
    for part in ("dev", "val"):
        DETS[(arm, part)] = load_detections(predictions_file(arm, part))
        REPORT[(arm, part)] = evaluate(GTS[part], DETS[(arm, part)], NAMES, SIZE_BINS, BOOTSTRAP, SEED)


def ap_table(part):
    rows = {}
    for arm in AVAIL + CONTEXT:
        s = REPORT[(arm, part)].summary
        ci = s["mAP50_ci95"]
        rows[arm] = {"mAP50": s["mAP50"], "CI95": f"[{ci[0]:.3f}, {ci[1]:.3f}]", "agnostic AP50": s["AP50_class_agnostic"],
                     **dict(zip(SHORT, REPORT[(arm, part)].per_class.set_index("class").AP50.reindex(NAMES)))}
    return pd.DataFrame(rows).T


TABLE = {part: ap_table(part) for part in ("dev", "val")}
for part in ("val", "dev"):
    print(part)
    display(TABLE[part].join(COMPOSITION[["kind", "budget %", "images", "instances"]]).round(3))

In [ ]:
REF_ARMS = [a for a in REFERENCE_ARMS if a in AVAIL]
REF = {part: recovery_threshold([REPORT[(a, part)].summary["mAP50"] for a in REF_ARMS], 0.9) for part in ("dev", "val")}
display(pd.DataFrame(REF).T)

rule = ARMS[(ARMS.kind != "full") & ARMS.available].copy()
for part in ("val", "dev"):
    rule[f"{part} mAP50"] = [REPORT[(a, part)].summary["mAP50"] for a in rule.index]
    rule[f"{part} % of full"] = 100 * rule[f"{part} mAP50"] / REF[part]["reference"]
rule["passes (val and dev)"] = (rule["val mAP50"] >= REF["val"]["threshold"]) & (rule["dev mAP50"] >= REF["dev"]["threshold"])
rule = rule.join(COMPOSITION[["images", "instances"]])
display(rule[["kind", "budget_frac", "images", "instances", "val mAP50", "val % of full", "dev mAP50", "dev % of full", "passes (val and dev)"]].round(3))
passing = rule[rule["passes (val and dev)"]]
if len(passing):
    best = passing.sort_values(["instances", "images"]).iloc[0]
    print(f"smallest tested subset that passes both: {best.name} ({int(best.instances)} instances = {100 * best.instances / N_FULL:.1f}% of the train_core labels, {int(best.images)} images)")
else:
    print("no tested subset passes both measures")

## Stage 4 · Coverage selection against random subsets of the same label budget

Coverage is deterministic and is run once per budget; random subsets are run once (subset seed 0) and, optionally, repeated at the smallest budget. With fewer than three random arms the random spread is not measured,
so a coverage gain is **not claimed** (amended protocol, declared before the runs); the paired bootstrap on the same images shows whether the difference is resolved at all, and the full-data seed spread (Stage 3) is the noise scale to read it against. Coverage keeps more images and, at the same label budget, a larger share of the rare classes; the per-class table
shows whether that is where its score comes from.

In [ ]:
rows = []
for p, f in BUDGET.items():
    rand = [f"SR{p}_k{k}" for k in (0, 1, 2) if f"SR{p}_k{k}" in AVAIL]
    cov = f"SC{p}"
    if not rand or cov not in AVAIL:
        continue
    for part in ("dev", "val"):
        r = [REPORT[(a, part)].summary["mAP50"] for a in rand]
        c = REPORT[(cov, part)].summary["mAP50"]
        rows.append({"budget %": 100 * f, "split": part, "random mean": np.mean(r), "random min": min(r), "random max": max(r),
                     "coverage": c, "above random range": c > max(r), "n random": len(r),
                     "claim allowed (3 random arms and above their range)": len(r) >= 3 and c > max(r)})
if rows:
    display(pd.DataFrame(rows).round(3))

for p in BUDGET:
    cov = f"SC{p}"
    for k in (0, 1, 2):
        rnd = f"SR{p}_k{k}"
        if cov in AVAIL and rnd in AVAIL:
            for part in ("dev", "val"):
                print(f"budget {100 * BUDGET[p]:g}%: coverage vs {rnd}, {part}")
                display(compare(GTS[part], DETS[(rnd, part)], DETS[(cov, part)], NAMES, BOOTSTRAP, SEED).round(3))

Subsets against the full-data reference (paired bootstrap on the same images, A = `D1_s0`, B = subset): the difference and its 95% interval show what the evaluation can and cannot resolve.

In [ ]:
base = "D1_s0"
if base in AVAIL:
    rows = []
    for arm in [a for a in AVAIL if ARMS.loc[a, "kind"] != "full"]:
        for part in ("dev", "val"):
            d = compare(GTS[part], DETS[(base, part)], DETS[(arm, part)], NAMES, BOOTSTRAP, SEED)
            for metric in ("mAP50", "AP50 class-agnostic"):
                rows.append({"arm": arm, "split": part, "metric": metric, "full": d.loc[metric, "A"], "subset": d.loc[metric, "B"],
                             "diff": d.loc[metric, "diff"], "CI95": f"[{d.loc[metric, 'lo']:+.3f}, {d.loc[metric, 'hi']:+.3f}]"})
    if rows:
        display(pd.DataFrame(rows).set_index(["split", "metric", "arm"]).sort_index().round(3))

## Stage 5 · The curve, the score per doubling of labels, and +500 labels (assignment 5.3)

`score = a + slope x log2(instances)`, fitted on the random arms and the full-data runs (coverage is shown, not fitted). The slope is the score per doubling; +500 labels on top of the 6,525 train_core instances
is 0.107 doublings. This is an extrapolation along a line fitted on a few noisy points: it is reported with the standard error of the slope, not as a prediction.

In [ ]:
inst = COMPOSITION.instances
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for k, part in enumerate(("dev", "val")):
    t = TABLE[part]
    for kind, style in (("random", dict(c="#4c72b0", marker="o", ls="")), ("coverage", dict(c="#dd8452", marker="s", ls="-")),
                        ("full", dict(c="k", marker="D", ls=""))):
        arms = [a for a in AVAIL if ARMS.loc[a, "kind"] == kind]
        if arms:
            ax[k].plot(inst[arms], t.loc[arms, "mAP50"], label=kind, **style)
    if REF[part]["n"]:
        ax[k].axhspan(REF[part]["min"], REF[part]["max"], color="k", alpha=0.08, label="full-data seeds (min-max)")
        ax[k].axhline(REF[part]["threshold"], c="r", ls="--", lw=1, label="90% of the full-data mean")
    ax[k].set(xscale="log", xlabel="labelled instances in the training subset", ylabel=f"{part} mAP50", title=f"{part}: mAP50 against label budget")
    ax[k].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fit_arms = [a for a in AVAIL if ARMS.loc[a, "kind"] in ("random", "full")]
fit_rows = []
for part in ("dev", "val"):
    for metric, col in (("mAP50", "mAP50"), ("class-agnostic AP50", "agnostic AP50")):
        if len(set(inst[fit_arms])) < 2:
            continue
        fit = log2_slope(inst[fit_arms].to_numpy(), TABLE[part].loc[fit_arms, col].to_numpy())
        gain = gain_for_extra_labels(fit["slope"], N_FULL, 500)
        fit_rows.append({"split": part, "metric": metric, "points": fit["n"], "slope per doubling": fit["slope"], "se": fit["se"],
                         "+500 labels": gain, "+500 as % of full score": 100 * gain / REF[part]["reference"]})
if fit_rows:
    display(pd.DataFrame(fit_rows).round(4))

# per class: dev AP50 of the class against the number of that class's instances in the subset (full data included)
cls_rows = []
for name, short in zip(NAMES, SHORT):
    n_c = np.array([COMPOSITION.loc[a, f"n {short}"] for a in fit_arms], float)
    ap = np.array([REPORT[(a, "dev")].per_class.set_index("class").AP50[name] for a in fit_arms], float)
    keep = (n_c > 0) & np.isfinite(ap)
    if len(set(n_c[keep])) < 2:
        continue
    fit = log2_slope(n_c[keep], ap[keep])
    n_full = float(COMPOSITION.loc[REFERENCE_ARMS[0], f"n {short}"])
    full_ap = np.nanmean([REPORT[(a, "dev")].per_class.set_index("class").AP50[name] for a in REF_ARMS])
    cls_rows.append({"class": short, "instances (full)": int(n_full), "AP50 dev (full)": full_ap, "slope per doubling": fit["slope"],
                     "se": fit["se"], "gain, +500 spread proportionally": fit["slope"] * np.log2(1 + 500 / N_FULL),
                     "gain, all +500 to this class (extrapolated)": gain_for_extra_labels(fit["slope"], n_full, 500)})
if cls_rows:
    display(pd.DataFrame(cls_rows).set_index("class").sort_values("slope per doubling", ascending=False).round(4))

## Stage 6 · What the surviving examples are

The composition of each subset (images, instances per class, images that contain the class, size mix) at every budget. The questions this stage has to answer from the numbers above
(fill in after the runs): does coverage keep the rare classes that random subsets lose; is its image count (scene diversity) or its class mix responsible for any difference; which classes lose most when the
budget shrinks; and whether a class with many instances but few distinct images (Tractor) behaves like a class with few instances.

In [ ]:
for p in BUDGET:
    arms = [a for a in COMPOSITION.index if a.startswith((f"SR{p}_", f"SC{p}"))]
    if arms:
        print(f"budget {100 * BUDGET[p]:g}%")
        display(COMPOSITION.loc[arms, ["images", "instances", "median inst/image", "xs %"] + [f"n {s}" for s in SHORT] + [f"img {s}" for s in SHORT]].round(1))

## Observations (fill in after the runs)

* **Reference:** full-data val and dev mAP50 over seeds 0 and 1 (mean, min, max), the 90% thresholds, and what the spread says about whether a 10% margin is resolvable.
* **Smallest tested subset that passes** (val and dev), its label budget as a share of train_core, its image count; the next smaller tested budget and its result. Not a proven minimum.
* **Random vs coverage** at each budget: difference against the random range and the paired intervals; per-class AP50 (is any gain in Flatbed/Tractor/Liquid?).
* **Curve shape (5.3):** slope per doubling overall and per class, +500 labels with its uncertainty; which classes benefit most and least; limited sample count versus label ambiguity.
* **Prediction check:** the predictions declared before the runs against what was observed.
* **Limits:** one training seed per subset arm, the fixed 50 epochs (fewer updates for small subsets), a 22-image val set, coverage deterministic and run once.